In [1]:
import pandas as pd
from io import StringIO

# Создание многострочной переменной с CSV данными
csv_data = """Пациент,Кашель (C),Температура (F),Затрудненное дыхание (B),Утомляемость (T),Диагноз (D)
1,X,X,X,X,Болен
2,X,,X,X,Болен
3,X,X,,X,Болен
4,X,X,X,,Болен
5,X,,,X,Здоров
6,,,X,,Здоров
7,,,,,Здоров
8,,,X,X,Здоров
"""

# Использование StringIO для имитации файла
csv_file_like_object = StringIO(csv_data)
df = pd.read_csv(csv_file_like_object)
df

,Пациент,Кашель (C),Температура (F),Затрудненное дыхание (B),Утомляемость (T),Диагноз (D)
0,1,X,X,X,X,Болен
1,2,X,NaN,X,X,Болен
2,3,X,X,NaN,X,Болен
3,4,X,X,X,NaN,Болен
4,5,X,NaN,NaN,X,Здоров
5,6,NaN,NaN,X,NaN,Здоров
6,7,NaN,NaN,NaN,NaN,Здоров
7,8,NaN,NaN,X,X,Здоров


In [2]:
df.rename(columns={
    'Пациент': 'id',
    'Кашель (C)': 'cough',
    'Температура (F)': 'fever',
    'Затрудненное дыхание (B)': 'breathing',
    'Утомляемость (T)': 'fatigue',
    'Диагноз (D)': 'diagnosis',
}, inplace=True)

df

,id,cough,fever,breathing,fatigue,diagnosis
0,1,X,X,X,X,Болен
1,2,X,NaN,X,X,Болен
2,3,X,X,NaN,X,Болен
3,4,X,X,X,NaN,Болен
4,5,X,NaN,NaN,X,Здоров
5,6,NaN,NaN,X,NaN,Здоров
6,7,NaN,NaN,NaN,NaN,Здоров
7,8,NaN,NaN,X,X,Здоров


In [4]:
pd.set_option('future.no_silent_downcasting', True)

df = df.replace({'X': 1})
df = df.replace({'Болен': 1, 'Здоров': 0})

df['cough'] = df['cough'].fillna(0)
df['fever'] = df['fever'].fillna(0)
df['breathing'] = df['breathing'].fillna(0)
df['fatigue'] = df['fatigue'].fillna(0)

df = df.set_index('id')
df = df.astype(int)

df

KeyError: "None of ['id'] are in the columns"

In [6]:
print(df)

    cough  fever  breathing  fatigue  diagnosis
id                                             
1       1      1          1        1          1
2       1      0          1        1          1
3       1      1          0        1          1
4       1      1          1        0          1
5       1      0          0        1          0
6       0      0          1        0          0
7       0      0          0        0          0
8       0      0          1        1          0


In [47]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Разделение данных на признаки и целевую переменную
X = df.drop('diagnosis', axis=1)
y = df['diagnosis']

# Разделение данных на обучающую и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

# Обучение модели логистической регрессии
model = LogisticRegression()
model.fit(X_train, y_train)

# Предсказания на тестовой выборке
y_pred = model.predict(X_test)

# Оценка модели
accuracy = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Точность: {accuracy}")
print("Матрица ошибок:\n", conf_matrix)
print("Отчет о классификации:\n", class_report)

Точность: 1.0
Матрица ошибок:
 [[1 0]
 [0 1]]
Отчет о классификации:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2



In [57]:
# Вывод весов признаков
feature_names = X.columns
weights = model.coef_[0]

for feature, weight in zip(feature_names, weights):
    print(f"Признак: {feature}, Вес: {weight}")

Признак: cough, Вес: 0.604469249161626
Признак: fever, Вес: 0.612274671948707
Признак: breathing, Вес: 0.6564899504584267
Признак: fatigue, Вес: 0.2697195880676966
